# Final Project: Steam Games Popularity Prediction

https://huggingface.co/datasets/FronkonGames/steam-games-dataset/tree/main

Authors: Autumn Mizer, Steven Thomas, Josiah Reinholz

### Research Questions

.. more specific but still viable
- What features are best at predicting a games popularity on Steam?
- Do meta critic reviews contribute to a games popularity?
- Does price affect the popularity of a game?
- Can playtime (2 weeks or overtime) suggest the popularity of a game?


### Hypothesis

H0 - There is not a relationship between a games features and its popularity on Steam

HA - There is a relationship between a games features and its popularity on Steam

### Dataset

Brief Summary of Dataset:
// todo dataset summary

Brief Summary for Model Plans:

We will use regression models to predict a games popularity on Steam. For our target variable we are going to use 'Estimated Owners', and test features against this variable. To see simple relationships we will do a model with linear regression, and then to further see combined features against the target we could do multiple linear regression models. We could also implement Random Forest in this to see non linear relationships with the data.

Since Estimated Owners is classified in bins, classification models could also be used for further testing. For regression use it will have to be changed to numeric

In [2]:
import pandas as pd
from sklearn.preprocessing import MultiLabelBinarizer, StandardScaler

df = pd.read_csv('games.csv')

print(df.shape)
df.head()

ModuleNotFoundError: No module named 'sklearn'

# Preprocessing 

In [24]:
# 1. See all unique owner brackets (the names of the classes)
unique_ranges = df['Estimated owners'].unique()
print("Unique Owner Brackets:")
print(unique_ranges)

# 2. See how many unique options there are (the count of classes)
num_options = df['Estimated owners'].nunique()
print(f"\nTotal number of unique categories: {num_options}")

# 3. See the distribution (how many games are in each bracket)
print("\nGames per category:")
print(df['Estimated owners'].value_counts())

# 4. (Optional) Sort them logically to see the progression
print("\nSorted categories:")
print(sorted(df['Estimated owners'].unique()))

Unique Owner Brackets:
['0 - 0' '0 - 20000' '100000 - 200000' '500000 - 1000000' '20000 - 50000'
 '200000 - 500000' '50000 - 100000' '2000000 - 5000000'
 '10000000 - 20000000' '1000000 - 2000000' '20000000 - 50000000'
 '5000000 - 10000000' '100000000 - 200000000' '50000000 - 100000000']

Total number of unique categories: 14

Games per category:
Estimated owners
0 - 20000                75404
0 - 0                    21641
20000 - 50000            11396
50000 - 100000            5355
100000 - 200000           3454
200000 - 500000           2853
500000 - 1000000          1154
1000000 - 2000000          729
2000000 - 5000000          405
5000000 - 10000000         125
10000000 - 20000000         51
20000000 - 50000000         31
50000000 - 100000000         9
100000000 - 200000000        4
Name: count, dtype: int64

Sorted categories:
['0 - 0', '0 - 20000', '100000 - 200000', '1000000 - 2000000', '10000000 - 20000000', '100000000 - 200000000', '20000 - 50000', '200000 - 500000', '2000000

In [ ]:
def preprocess_steam_data(df):
    # 1. CLEAN TARGET VARIABLE (Estimated Owners)
    owner_map = {
        "0 - 0": 0,
        "0 - 20000": 1,
        "20000 - 50000": 2,
        "50000 - 100000": 3,
        "100000 - 200000": 4,
        "200000 - 500000": 5,
        "500000 - 1000000": 6,
        "1000000 - 2000000": 7,
        "2000000 - 5000000": 8,
        "5000000 - 10000000": 9,
        "10000000 - 20000000": 10,
        "20000000 - 50000000": 11,
        "50000000 - 100000000": 12,
        "100000000 - 200000000": 13,
    }
    df['target'] = df['Estimated owners'].map(owner_map)
    
    # 2. DATE ENGINEERING
    # Convert 'Aug 1, 2023' format to datetime
    df['Release date'] = pd.to_datetime(df['Release date'], errors='coerce')
    # Calculate days since release (relative to "today" in 2026)
    reference_date = pd.to_datetime('2026-04-30')
    df['Days_Since_Release'] = (reference_date - df['Release date']).dt.days.fillna(0)

    # 3. NUMERIC FEATURES & RATIOS
    # Create a success ratio (Positive reviews vs Total)
    df['Total_Reviews'] = df['Positive'] + df['Negative']
    df['Positive_Ratio'] = df['Positive'] / df['Total_Reviews']
    df['Positive_Ratio'] = df['Positive_Ratio'].fillna(0) # Handle games with 0 reviews
    
    # 4. BINARY ENCODING (True/False to 1/0)
    for col in ['Windows', 'Mac', 'Linux']:
        df[col] = df[col].astype(int)

    # 5. MULTI-LABEL ENCODING (Genres & Tags)
    # These columns are strings like "Action,Adventure,RPG"
    def split_and_clean(x):
        if pd.isna(x) or x == "": return []
        return [i.strip() for i in str(x).split(',')]

    mlb = MultiLabelBinarizer()
    
    # Process Genres
    genre_series = df['Genres'].apply(split_and_clean)
    genre_encoded = mlb.fit_transform(genre_series)
    genre_df = pd.DataFrame(genre_encoded, columns=[f"Genre_{c}" for c in mlb.classes_])
    
    # 6. Convert Text to Length
    df['About_Length'] = df['About the game'].str.len().fillna(0)

    # 7. FINAL ASSEMBLY
    # Drop high-cardinality/text columns that aren't useful for basic classification
    cols_to_drop = [
        'AppID', 'Name', 'Estimated owners', 'Release date', 'About the game', 
        'Supported languages', 'Full audio languages', 'Reviews', 'Header image', 
        'Website', 'Support url', 'Support email', 'Metacritic url', 'Developers', 
        'Publishers', 'Categories', 'Genres', 'Tags', 'Screenshots', 'Movies', 'Notes'
    ]
    
    df_final = df.drop(columns=cols_to_drop)
    df_final = pd.concat([df_final, genre_df], axis=1)
    
    return df_final

In [27]:
df_clean = preprocess_steam_data(df)
df_clean.head()

,Peak CCU,Required age,Price,Discount,DLC count,Windows,Mac,Linux,Metacritic score,User score,...,Genre_Short,Genre_Simulation,Genre_Software Training,Genre_Sports,Genre_Strategy,Genre_Tutorial,Genre_Utilities,Genre_Video Production,Genre_Violent,Genre_Web Publishing
0,0,0,0.00,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,5.24,65,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,4.99,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,8.99,0,1,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,0,0,4.99,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [ ]:
numeric_cols = [
    "Price", "Peak CCU", "Positive_Ratio", "Negative_Ratio", "Days_Since_Release",
    "Metacritic score", "Recommendations", "Average playtime forever",
    "Achievements", "Required age", "About_Length"
]
scaler = StandardScaler()
df_clean[numeric_cols] = scaler.fit_transform(df_clean[numeric_cols])

df_clean.head()

,Peak CCU,Required age,Price,Discount,DLC count,Windows,Mac,Linux,Metacritic score,User score,...,Genre_Short,Genre_Simulation,Genre_Software Training,Genre_Sports,Genre_Strategy,Genre_Tutorial,Genre_Utilities,Genre_Video Production,Genre_Violent,Genre_Web Publishing
0,-0.014638,0,-0.380265,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,-0.014638,0,0.037899,65,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,-0.014638,0,0.017948,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,-0.014370,0,0.337157,0,1,1,0,0,0,0,...,0,1,0,0,0,0,0,0,0,0
4,-0.014638,0,0.017948,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### Statistical Analysis

In [ ]:
kruskal_results = []

for var in numeric_cols:
    x = df_test[var].values
    y = df_test['target'].values

    mask = ~np.isnan(x) & ~np.isnan(y)
    data_x = x[mask]
    data_y = y[mask]

    samples_by_group = []
    for value in set(data_y):
        samples_by_group.append(data_x[data_y == value])

    stat, p = kruskal(*samples_by_group)
    n = len(data_x)
    k = len(set(data_y))
    epsilon_sq = (stat - k + 1) / (n - k)   # epsilon-squared effect size

    kruskal_results.append({
        'variable': var,
        'test': 'Kruskal-Wallis',
        'statistic': stat,
        'p_value': p,
        'epsilon_squared': epsilon_sq
    })

pd.DataFrame(kruskal_results)

NameError: name 'numeric_cols' is not defined

In [7]:
from scipy.stats import chi2_contingency

categorical_variables = ["Windows", "Mac", "Linux"] + [col for col in df_test.columns if col.startswith("Genre_")]

alpha = 0.05
bonferroni_alpha = alpha / len(categorical_variables)

chi2_results = []

for var in categorical_variables:
    x = df_test[var].astype(float).values
    y = df_test["target"].values

    mask = ~np.isnan(x) & ~np.isnan(y)
    data_x = x[mask]
    data_y = y[mask]

    ct = pd.crosstab(data_y, data_x)
    chi2_stat, p, dof, expected = chi2_contingency(ct)

    n = ct.sum().sum()
    min_dim = min(ct.shape) - 1
    cramers_v = np.sqrt(chi2_stat / (n * min_dim))

    chi2_results.append({
        'variable': var,
        'test': 'Chi-Squared',
        'statistic': chi2_stat,
        'p_value': p,
        'significant (Bonferroni)': p < bonferroni_alpha
    })

pd.DataFrame(chi2_results)

ModuleNotFoundError: No module named 'scipy'

In [8]:
import matplotlib.pyplot as plt

owner_order = [
    "0 - 0", "0 - 20000", "20000 - 50000", "50000 - 100000",
    "100000 - 200000", "200000 - 500000", "500000 - 1000000",
    "1000000 - 2000000", "2000000 - 5000000", "5000000 - 10000000",
    "10000000 - 20000000", "20000000 - 50000000", "50000000 - 100000000",
    "100000000 - 200000000"
]

counts = df["Estimated owners"].value_counts().reindex(owner_order)

plt.figure(figsize=(12, 5))
plt.bar(owner_order, counts)
plt.xticks(rotation=45, ha="right")
plt.xlabel("Estimated Owners")
plt.ylabel("Number of Games")
plt.title("Distribution of Estimated Owners")
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

In [9]:
import seaborn as sns
import matplotlib.pyplot as plt



fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(14, 5 * len(numeric_cols)))

for i, var in enumerate(numeric_cols):
    sns.violinplot(data=df_clean, x="target", y=var, ax=axes[i])
    axes[i].set_title(f"{var} by Estimated Owners Group")
    axes[i].set_xlabel("Ownership Group (1 = 0-20K, 13 = 100M+)")
    axes[i].set_ylabel(var)

plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'seaborn'

In [10]:
import matplotlib.pyplot as plt

variables = [r['variable'] for r in kruskal_results]
epsilon_values = [r['epsilon_squared'] for r in kruskal_results]

plt.figure(figsize=(10, 6))
bars = plt.barh(variables, epsilon_values)
plt.xlabel('Epsilon Squared (Effect Size)')
plt.title('Kruskal-Wallis Effect Size by Variable')
plt.tight_layout()
plt.show()

ModuleNotFoundError: No module named 'matplotlib'